# 04 — Birkhoff and the Assignment Problem

Close the Monge→Kantorovich arc with its most elegant special case — uniform marginals, where transport becomes the classical assignment problem — and open the transport row of the classical↔quantum dictionary.

**Prerequisites:** `01_the_monge_problem`, `02_where_monge_breaks`, `03_the_kantorovich_lp`.

**What you'll be able to do**
- Recognise the Birkhoff polytope of doubly stochastic matrices as the uniform-marginal transportation polytope.
- Use the Birkhoff–von Neumann theorem to connect optimal transport to the assignment problem.
- Read the first transport rows of the running classical↔quantum dictionary.

You have built the whole arc: you posed the transport problem and Monge's map (01), watched the map break when mass must split (02), and relaxed it into the Kantorovich linear program over couplings (03). This synthesis closes the arc with the cleanest case of all — uniform, equal-size marginals — where the polytope's corners are permutations and transport reduces to the assignment problem. And here the classical↔quantum dictionary opens its transport row, the thread we follow all the way to quantum optimal transport.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.transport.discrete import (
    squared_euclidean_cost,
    discrete_ot_plan,
    is_doubly_stochastic,
)

np.random.seed(42)
viz.use_course_style()

## Where we are

Three bricks brought us here: `01_the_monge_problem` made transport precise and gave Monge's deterministic map; `02_where_monge_breaks` showed that map failing when one source must feed two targets; `03_the_kantorovich_lp` relaxed the map into a coupling and solved the problem as a linear program. Now we ask what happens in the cleanest case — and collect the dictionary row the arc has earned.

## 1. The Birkhoff polytope and the assignment problem

Specialise to **uniform, equal-size** marginals $a = b = \mathbf{1}_n / n$. Then, up to a factor $n$, the transportation polytope becomes the **Birkhoff polytope**

$$ B_n = \{ M \in \mathbb{R}_+^{n \times n} :\ M\mathbf{1} = \mathbf{1},\ M^\top\mathbf{1} = \mathbf{1} \}, $$

the set of **doubly stochastic** matrices. Its structure is captured by a classic theorem:

> **Birkhoff–von Neumann (1946).** The extreme points of $B_n$ are exactly the $n!$ **permutation matrices**. Every doubly stochastic matrix is a convex combination of permutations.

Because a linear program attains its optimum at a vertex, running Kantorovich on uniform equal-size data returns a **permutation** — a bijection matching each source to one target. Transport collapses to the **assignment problem**

$$ \min_{\sigma \in \mathcal{S}_n}\ \sum_{i=1}^n C_{i,\sigma(i)}, $$

solved in $O(n^3)$ by the Hungarian algorithm. In this special case, Monge's deterministic dream comes true after all.

In [ ]:
# Uniform, equal-size marginals: source at [0, 1, 2], target at [3, 1, 5], mass 1/3 each.
x_src = np.array([0.0, 1.0, 2.0])
x_tgt = np.array([3.0, 1.0, 5.0])
a = np.full(3, 1 / 3)
b = np.full(3, 1 / 3)

C = squared_euclidean_cost(x_src, x_tgt)
plan = discrete_ot_plan(a, b, C)
perm_matrix = 3 * plan   # scale by n = 3: expect a 0/1 permutation matrix

print("optimal plan (uniform, equal-size) =")
print(np.round(plan, 3))
print(f"\n3 x plan (expect a 0/1 permutation matrix) =")
print(np.round(perm_matrix, 3))
print(f"\ndoubly stochastic (3 x plan)? {is_doubly_stochastic(perm_matrix)}")
print(f"all entries in {{0, 1}}?        {bool(np.all(np.isin(np.round(perm_matrix), [0, 1])))}")

viz.plot_transport_arrows(
    a, b, plan,
    source_positions=x_src, target_positions=x_tgt,
    title="Uniform equal-size OT: the optimal plan is a permutation (the assignment problem)",
)
plt.show()

**Read the output and flow view.** The optimal plan concentrates on three cells out of nine, each equal to $1/3$ — a permutation scaled by $1/3$. Multiplying by $n = 3$ recovers a clean $0/1$ permutation, confirmed doubly stochastic. In the flow view, every source sends *all* of its mass to one target: no splitting is needed here. The match follows sorted rank — source rank-$k$ to target rank-$k$ once both are ordered by position — a one-dimensional fact we return to in `06_1d_quantile_closed_form`.

## 2. Birkhoff decomposition — a mixture of permutations

The theorem's converse is constructive: any doubly stochastic matrix can be *written* as a convex combination of permutation matrices. We show it on an explicit interior point.

In [ ]:
identity = np.eye(3)
cycle = np.array([[0, 1, 0], [0, 0, 1], [1, 0, 0]], dtype=float)  # the 3-cycle 1 -> 2 -> 3 -> 1
interior = 0.5 * identity + 0.5 * cycle

print("interior doubly stochastic matrix =")
print(interior)
print(f"doubly stochastic?                {is_doubly_stochastic(interior)}")
print(f"reconstructs as 0.5*I + 0.5*cycle? {np.allclose(interior, 0.5 * identity + 0.5 * cycle)}")

fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
panels = [
    (identity, "permutation P1 = I"),
    (cycle, "permutation P2 (3-cycle)"),
    (interior, "0.5 P1 + 0.5 P2  (doubly stochastic)"),
]
for ax, (matrix, name) in zip(axes, panels):
    im = ax.imshow(matrix, cmap=viz.CMAP_PLAN, vmin=0, vmax=1, origin="lower")
    ax.set_title(name, pad=10)
    ax.set_xticks(range(3))
    ax.set_yticks(range(3))
    ax.grid(False)
    fig.colorbar(im, ax=ax, shrink=0.85)
fig.suptitle("Birkhoff-von Neumann: doubly stochastic matrices are the convex hull of permutations",
             fontsize=13, fontweight="bold", y=1.05)
plt.tight_layout()
plt.show()

**Read the three matrices.** The first two panels are the permutations $I$ and a 3-cycle. Their half-and-half combination (right) is doubly stochastic but **not** a permutation — its entries are $0$ and $1/2$, not $0$ and $1$. Every interior point of $B_n$ admits such a decomposition. This vertex structure is the geometric reason the Kantorovich LP is so tractable: its optima live at the corners, and the corners are clean combinatorial objects.

## 3. Dictionary update — the transport row opens

Module 2 filled the *information* rows of our running classical↔quantum dictionary. The Monge→Kantorovich arc now adds the **transport** rows: a coupling is the classical shadow of a bipartite quantum state, the Kantorovich LP of a semidefinite program, the transportation polytope of the set of quantum couplings. Every later synthesis extends this table; today it earns its first transport entries.

| Classical | Quantum |
|-----------|---------|
| probability vector $a$ on positions $\{x_i\}$ | density matrix $\rho$ |
| marginal | partial trace |
| Shannon entropy $H(p)$ | von Neumann entropy $S(\rho)$ |
| KL divergence $D(p\|q)$ | Umegaki relative entropy $S(\rho\|\sigma)$ |
| Fisher–Rao metric | Bures / quantum Fisher metric |
| **transport plan / coupling $P \in T(a, b)$** | **quantum coupling**: bipartite $\rho_{AB}$ with $\mathrm{tr}_B\,\rho_{AB} = \rho_A$, $\mathrm{tr}_A\,\rho_{AB} = \rho_B$  *(Module 4)* |
| **Kantorovich LP** $\min \langle C, P\rangle$ | **quantum OT SDP** $\min \mathrm{tr}(C\,\rho_{AB})$  *(Module 4)* |
| **transportation polytope** $T(a, b)$ | **set of quantum couplings**  *(Module 4)* |
| **Birkhoff polytope / assignment problem** | deterministic-coupling case  *(Module 4 capstone)* |

The classical transport row now exists. The next arc pulls this *cost* into a genuine *distance*.

## Your turn

1. **Count the vertices.** A basic feasible solution of $T(a, b)$ has at most $n + m - 1$ non-zero entries. Check this on the uniform optimum above (where it equals $5$, yet the permutation uses only $3$). Why can the optimum use *fewer* than the bound?
2. **Reverse-decompose.** Build your own $3 \times 3$ doubly stochastic matrix and write it as an explicit convex combination of two permutation matrices. (A general member of $B_n$ needs at most $n^2 - 2n + 2$ permutations — Marcus–Newman.)
3. **When is splitting unavoidable?** The uniform case gave a clean permutation. Perturb one mass so the marginals are no longer equal-size, re-solve, and watch a row split again. What property of the marginals decides whether a pure assignment can win?

## What you built

- You recognised the Birkhoff polytope of doubly stochastic matrices as the uniform-marginal transportation polytope.
- You used the Birkhoff–von Neumann theorem to link optimal transport to the assignment problem, and watched the optimum become a permutation.
- You opened the transport row of the classical↔quantum dictionary.

That completes the Monge→Kantorovich arc: from a deterministic dream, through its failure, to a linear program and its elegant special case. You can now pose, solve, and structurally understand discrete optimal transport — the foundation everything else in this course rests on.

## What's next

The Kantorovich LP gives a *cost*. In `05_wasserstein_distance` we turn that cost into a genuine **metric** — the Wasserstein distance — and find that it respects the geometry of the ground space in a way the information divergences of Module 2 could not.

## References

- G. Monge, "Mémoire sur la théorie des déblais et des remblais", *Histoire de l'Académie Royale des Sciences de Paris* (1781).
- L. V. Kantorovich, "On the translocation of masses", *Doklady Akademii Nauk SSSR* 37, 199–201 (1942).
- G. Birkhoff, "Three observations on linear algebra", *Univ. Nac. Tucumán Rev. A* 5, 147–151 (1946).
- G. Peyré & M. Cuturi, *Computational Optimal Transport*, NoW (2019), chs. 2–3. DOI:10.1561/2200000073
- C. Villani, *Topics in Optimal Transportation*, AMS (2003), ch. 1.

**Previous:** `notebooks/03_ClassicalOptimalTransport/03_the_kantorovich_lp.ipynb` · **Next:** `notebooks/03_ClassicalOptimalTransport/05_wasserstein_distance.ipynb`